# NyayaAI - Colab Training Notebook

End-to-end pipeline for **IndiaLex-FT** — a LoRA fine-tuned LLM for Indian legal and income-tax Q&A.

**Steps covered:**
1. Clone repo and install dependencies
2. Load API keys from Colab Secrets
3. Baseline evaluation (pre-training)
4. LoRA SFT fine-tuning
5. Post-training evaluation
6. LLM-as-Judge scoring via claude-sonnet-4-6
7. Download outputs

> **Runtime:** Select `Runtime → Change runtime type → T4 GPU` before starting.

In [ ]:
!git clone https://github.com/Harshith2014/Nyaya-AI.git
%cd Nyaya-AI/indialex-ft

In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
import os
from google.colab import userdata

def _load_secret(name, required=True):
    try:
        val = userdata.get(name)
        os.environ[name] = val
        print(f"{name} loaded: True")
    except Exception:
        if required:
            print(f"{name} loaded: MISSING (required — add it in Secrets)")
        else:
            print(f"{name} loaded: skipped (optional)")

_load_secret("ANTHROPIC_API_KEY", required=True)
_load_secret("GROQ_API_KEY",      required=True)
_load_secret("WANDB_API_KEY",     required=False)
_load_secret("HF_TOKEN",          required=False)

In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("HuggingFace login successful.")
else:
    print("HF_TOKEN not set — skipping HuggingFace login (only needed for gated models).")

## Step 1: Baseline Evaluation

Runs inference on the test split using the **base model** (before fine-tuning).
Computes ROUGE-L and BERTScore F1, and saves results to `evals/baseline_results.json`.

In [ ]:
!python scripts/baseline_eval.py

## Step 2: Training

Fine-tunes the base model with **LoRA + QLoRA (4-bit NF4)** via TRL SFTTrainer.
Saves the LoRA adapter to `outputs/adapter/` and the merged model to `outputs/merged/`.
Logs metrics to Weights & Biases.

In [ ]:
!python scripts/train.py

## Step 3: Post-training Evaluation

Runs inference on the test split using the **fine-tuned merged model**.
Computes ROUGE-L and BERTScore F1, compares against baseline, and saves
a delta summary to `evals/ft_results.json`.

In [ ]:
!python scripts/evaluate.py

## Step 4: LLM-as-Judge Evaluation

Uses **claude-sonnet-4-6** (Anthropic API) to score each fine-tuned answer on
four legal-domain dimensions (1–5 each): correctness, faithfulness, helpfulness,
and legal accuracy. Results saved to `evals/judge_results.json`.

In [ ]:
!python evals/llm_judge.py --results evals/ft_results.json

In [ ]:
import shutil

shutil.make_archive("nyayaai_outputs", "zip", root_dir=".", base_dir="outputs")
print("Zipped outputs/ → nyayaai_outputs.zip")

shutil.make_archive("nyayaai_evals", "zip", root_dir=".", base_dir="evals")
print("Zipped evals/   → nyayaai_evals.zip")

try:
    from google.colab import files
    files.download("nyayaai_outputs.zip")
    files.download("nyayaai_evals.zip")
except ImportError:
    print("Not running in Colab — skipping auto-download.")
    print("Zip files saved locally: nyayaai_outputs.zip, nyayaai_evals.zip")